# Notebook 1 — Validate Your Environment

Run every cell top to bottom. Each check prints ✅ or ❌.
All checks must pass before continuing to Notebook 2.

In [ ]:
# ── Cell 0: Sync lab materials from GitHub ────────────────────────────────────
# Clones (or updates) the lab repo so all notebooks, images and model weights
# are available locally with NO manual oc cp needed.
import subprocess, os, pathlib

REPO_URL   = 'https://github.com/faheemshai/1512_model_training.git'
LOCAL_PATH = os.path.expanduser('~/lab-materials')

if pathlib.Path(LOCAL_PATH, '.git').is_dir():
    print('Repo already cloned — pulling latest...')
    r = subprocess.run(['git', '-C', LOCAL_PATH, 'pull', '--ff-only'],
                       capture_output=True, text=True)
    print(r.stdout.strip() or 'Already up to date.')
else:
    print('Cloning lab repo (first time, ~10 s)...')
    r = subprocess.run(['git', 'clone', REPO_URL, LOCAL_PATH],
                       capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip())

LAB = LOCAL_PATH
print(f'✅ Lab materials ready at: {LAB}')

In [ ]:
import os, sys

NAMESPACE = open('/var/run/secrets/kubernetes.io/serviceaccount/namespace').read().strip()
print(f'Your namespace : {NAMESPACE}')
assert NAMESPACE.startswith('student'), f'❌ Expected student namespace, got: {NAMESPACE}'

In [ ]:
# ── S3 credentials (injected by the data connection) ─────────────────────────
S3_ENDPOINT   = os.environ.get('AWS_S3_ENDPOINT',       'http://s3.openshift-storage.svc:80')
S3_ACCESS_KEY = os.environ.get('AWS_ACCESS_KEY_ID',     '')
S3_SECRET_KEY = os.environ.get('AWS_SECRET_ACCESS_KEY', '')
S3_BUCKET     = os.environ.get('AWS_S3_BUCKET',         '')
S3_REGION     = os.environ.get('AWS_DEFAULT_REGION',    'us-east-1')
print(f'S3 endpoint : {S3_ENDPOINT}')
print(f'S3 bucket   : {S3_BUCKET}')
print(f'Access key  : {S3_ACCESS_KEY[:6]}... (truncated)')

In [ ]:
# ── Check 1: Core packages pre-installed ─────────────────────────────────────
import boto3, torch, ultralytics, onnxruntime
print(f'✅ boto3        : {boto3.__version__}')
print(f'✅ torch        : {torch.__version__}')
print(f'✅ ultralytics  : {ultralytics.__version__}')
print(f'✅ onnxruntime  : {onnxruntime.__version__}')
print(f'   CUDA         : {torch.cuda.is_available()} (CPU-only is fine)')

In [ ]:
# ── Check 2: S3 / NooBaa reachable ───────────────────────────────────────────
from botocore.client import Config
s3 = boto3.client(
    's3',
    endpoint_url          = S3_ENDPOINT,
    aws_access_key_id     = S3_ACCESS_KEY,
    aws_secret_access_key = S3_SECRET_KEY,
    region_name           = S3_REGION,
    config                = Config(signature_version='s3v4'),
    verify                = False
)
try:
    s3.head_bucket(Bucket=S3_BUCKET)
    print(f'✅ S3 / NooBaa reachable — bucket "{S3_BUCKET}" exists')
except Exception as e:
    try:
        s3.list_buckets()
        print(f'✅ S3 reachable (bucket "{S3_BUCKET}" will be created on first upload)')
    except Exception as e2:
        print(f'❌ S3 not reachable: {e2}')

In [ ]:
# ── Check 3: Sample images present (from git repo) ───────────────────────────
import pathlib
IMG_DIR = pathlib.Path(LAB) / 'sample-images'
all_ok  = True
for fname in ['cardboard_box.jpg', 'damaged_package.jpg', 'street_scene.jpg']:
    p = IMG_DIR / fname
    if p.exists():
        print(f'  ✅ {fname}  ({p.stat().st_size/1024:.0f} KB)')
    else:
        print(f'  ❌ {fname}  NOT FOUND at {p}')
        all_ok = False
if not all_ok:
    raise FileNotFoundError('Sample images missing — re-run the git clone cell above.')

In [ ]:
# ── Check 4: YOLO weights present (from git repo) ────────────────────────────
import pathlib
weights = pathlib.Path(LAB) / 'models' / 'yolov8n-cls.pt'
if weights.exists():
    print(f'✅ yolov8n-cls.pt  ({weights.stat().st_size/1024/1024:.1f} MB) at {weights}')
else:
    print(f'❌ yolov8n-cls.pt NOT found — re-run git clone cell above.')

In [ ]:
# ── Check 5: S3 read/write ────────────────────────────────────────────────────
import json
s3.put_object(Bucket=S3_BUCKET, Key='lab-test/ping.json',
              Body=json.dumps({'status':'ok','namespace':NAMESPACE}).encode())
data = json.loads(s3.get_object(Bucket=S3_BUCKET, Key='lab-test/ping.json')['Body'].read())
assert data['namespace'] == NAMESPACE
print('✅ S3 read/write working')

In [ ]:
print('\n══════════════════════════════════')
print(' Environment validation complete')
print('══════════════════════════════════')
print(f' Namespace : {NAMESPACE}')
print(f' S3 bucket : {S3_BUCKET}')
print(f' Lab path  : {LAB}')
print('\n✅ All checks passed — proceed to Notebook 2')